#Transformar Dados de Drivers

- 1 - Ler a tabela drivers da camada bronze
- 2 - Manter apenas as colunas necessárias para análise (remover a coluna url)
- 3 - Padronizar os nomes das colunas usando snake_case (driverId → driver_id, date0fbirth → date_of_birth)
- 4 - Concatenar name.givenName e name.familyName para criar uma nova coluna chamada driver_name e transformar o valor para Title Case.
- 5 - Remover registros duplicados
- 6 - Transformar os valores das colunas nationality para Title Case
- 7 - Escrever os dados transformados na tabela constructors da camada silver 

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

1 - Ler a tabela drivers da camada bronze

In [0]:
drivers_df = (
    spark.table(bronze_table).filter((F.col("batch_id")== v_batch_id))
               )

2 - Manter apenas as colunas necessárias para análise (remover a coluna url)

In [0]:
drivers_drop_df = drivers_df.drop(F.col("url"))

3 - Padronizar os nomes das colunas usando snake_case (driverId → driver_id, date0fbirth → date_of_birth)

In [0]:
drivers_renamed_df = drivers_drop_df.withColumnsRenamed({
    "driverId": "driver_id",
    "dateOfBirth" : "date_of_birth"
})

In [0]:
display(drivers_renamed_df.select("name"))

4 - Concatenar name.givenName e name.familyName para criar uma nova coluna chamada driver_name e transformar o valor para Title Case.

In [0]:
 drivers_concatenated_df = (
     drivers_renamed_df
     .withColumn("driver_name", 
                 F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
     .drop("name")
 )

In [0]:
display(drivers_concatenated_df)

In [0]:
drivers_duplicates_df = drivers_concatenated_df.dropDuplicates(["driver_id"])


In [0]:
drivers_final_df = ( drivers_duplicates_df
                    .withColumn('nationality', F.initcap(F.col("nationality")))
                    )

In [0]:
write_to_silver(
    input_df=drivers_final_df,
    target_table=silver_table,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
      "date_of_birth",
      "nationality",
       "ingestion_timestamp",
       "source_file",
        "batch_id",
        "driver_name"
    ]
)

In [0]:
display(spark.table(silver_table))

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name,created_timestamp,updated_timestamp
ahrens,1940-04-19,German,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Kurt Ahrens,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
barilla,1961-04-20,Italian,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Paolo Barilla,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
bayol,1914-02-28,French,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Élie Bayol,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
birger,1924-01-07,Argentine,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Pablo Birger,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
borgudd,1946-11-25,Swedish,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Slim Borgudd,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
branca,1916-09-15,Swiss,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Toni Branca,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
brown,1949-12-24,Australian,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Warwick Brown,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
christie,1924-04-04,American,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Bob Christie,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
clark,1936-03-04,British,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,Jim Clark,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
george_connor,1906-08-16,American,2026-09-11T22:55:00.805Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/drivers.json,2025-01,George Connor,2026-09-12T16:04:49.395Z,2026-09-12T16:04:49.395Z
